# Praktikum Komputasi 5: Simulasi Persamaan Laplace 2D (Julia)

Persamaan Laplace 2D memodelkan distribusi potensial elektrostatik $V(x,y)$ di dalam ruang bebas muatan antar-konduktor (misalnya di antara dua pelat kapasitor atau penampang kabel koaksial):
$$ \nabla^2 V = \frac{\partial^2 V}{\partial x^2} + \frac{\partial^2 V}{\partial y^2} = 0 $$

## Metode Relaksasi Beda Hingga (Metode Gauss-Seidel / Jacobi)
Dengan diskritisasi beda pusat spasial $\Delta x = \Delta y = h$:
$$ \frac{V_{i+1,j} - 2V_{i,j} + V_{i-1,j}}{h^2} + \frac{V_{i,j+1} - 2V_{i,j} + V_{i,j-1}}{h^2} = 0 $$

Menghasilkan formula rata-rata 4 titik tetangga (*Mean-Value Property*):
$$ V_{i,j}^{(k+1)} = \frac{1}{4} \left( V_{i+1,j}^{(k)} + V_{i-1,j}^{(k)} + V_{i,j+1}^{(k)} + V_{i,j-1}^{(k)} \right) $$

In [ ]:
using Plots

## 1. Implementasi Algoritma Relaksasi Gauss-Seidel 2D

In [ ]:
# Ukuran grid penampang 2D
Nx, Ny = 60, 60
V = zeros(Nx, Ny)

# Syarat Batas Dirichlet:
# Batas Atas (y = max): Pelat konduktor bertegangan 100 Volt
# Batas Kiri, Kanan, Bawah: Ground (0 Volt)
V[:, Ny] .= 100.0   # Batas atas
V[:, 1]  .= 0.0     # Batas bawah
V[1, :]  .= 0.0     # Batas kiri
V[Nx, :] .= 0.0     # Batas kanan

# Parameter Iterasi Relaksasi
max_iter = 2000
toleransi = 1e-4
iter_selesai = 0

for iter in 1:max_iter
    max_selisih = 0.0
    for i in 2:(Nx-1)
        for j in 2:(Ny-1)
            V_baru = 0.25 * (V[i+1, j] + V[i-1, j] + V[i, j+1] + V[i, j-1])
            selisih = abs(V_baru - V[i, j])
            if selisih > max_selisih
                max_selisih = selisih
            end
            V[i, j] = V_baru
        end
    end
    
    if max_selisih < toleransi
        iter_selesai = iter
        break
    end
    iter_selesai = iter
end

println("Konvergensi tercapai pada iterasi ke-$iter_selesai dengan toleransi $toleransi V")

## 2. Visualisasi Kontur Potensial Elektrostatik & Garis Ekipotensial

In [ ]:
x_range = range(0, 1, length=Nx)
y_range = range(0, 1, length=Ny)

# Heatmap Kontur 2D
p1 = contourf(collect(x_range), collect(y_range), V', color=:turbo, levels=20, 
              xlabel="Posisi X (m)", ylabel="Posisi Y (m)",
              title="Distribusi Potensial Elektrostatik V(x,y)")
display(p1)

## 3. Visualisasi Permukaan 3D (Potential Well)

In [ ]:
surface(collect(x_range), collect(y_range), V', xlabel="X", ylabel="Y", zlabel="Volt (V)",
        title="Permukaan Potensial Elektrostatik 3D", color=:viridis)

## 4. Analisis & Evaluasi Mahasiswa
1. **Medan Listrik $\mathbf{E}$:** Ingat bahwa medan listrik adalah gradien negatif dari potensial: $\mathbf{E} = -\nabla V$. Di area manakah kerapatan garis kontur paling rapat (artinya medan listrik $\mathbf{E}$ paling kuat)?
2. **Solusi Analitik Deret Fourier:** Solusi analitik untuk kasus ini adalah deret Fourier sinus tak hingga:
   $$ V(x,y) = \frac{4V_0}{\pi} \sum_{n=1,3,5,...}^{\infty} \frac{1}{n} \frac{\sinh(n\pi y / L)}{\sinh(n\pi)} \sin\left(\frac{n\pi x}{L}\right) $$
   Bandingkan nilai titik tengah $V(0.5, 0.5)$ numerik dengan solusi deret 5 suku pertama!